# 14 — V6-A: hierarchical-sampling TCN

This notebook implements **only the first post-v5 experiment** from
`ml/EXPERIMENT_BACKBONE.md`:

1. preserve the frozen v5 source-group split for comparable evidence;
2. retire `plank` by mapping it to `unknown` after the split is reconstructed;
3. sample hierarchically instead of letting long videos dominate;
4. apply Notebook 13 hard-positive weights only to matching original training
   windows;
5. keep the v5 two-stage TCN, 64-frame input, optimizer, and interim
   validation-only threshold sweep unchanged.

For the gate, sampling is `supported/unknown -> source group -> window`. For the
exercise classifier it is `exercise class -> source group -> window`. Validation
and test loaders are never resampled or reweighted.

Use a **T4 runtime**. The notebook downloads the seven existing landmark
archives and the small Notebook 13 CSV from GitHub. It does not rerun MediaPipe.
The output is `calorie_v6a_hierarchical_tcn_report.zip`.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
LEGACY_COORDINATE_NOISE_STD = 0.01  # held constant from v5; V6-B expands augmentation
RETIRED_CLASSES = {"plank"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Load frozen landmark evidence and Notebook 13 weights

No video is downloaded and MediaPipe does not run here. The hard-positive CSV
contains `window_key` values for difficult supported windows mined from the v5
**training split only**. It supplies weights, not copied training examples.


In [ ]:
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    "feat/tcn-rebuild/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_unknown_v1.zip"),
    ("v2", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5", "calorie_workoutfitness_gate_hardneg_v1.zip"),
]


def download(url, destination):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for version, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(f"{REPOSITORY_RAW}/{version}/landmarks/{name}", destination)
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH,
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique, "Hard-positive window keys must be unique."

SOURCE_ROOT = Path("/content/v6a_landmark_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Reconstruct v5 windows and freeze the split

The split is computed using the original v5 labels and ordering. Only after
those group assignments are fixed does V6-A map `plank` to `unknown` and remove
unreviewed groups. This preserves every remaining group's historical split and
prevents accidental validation/test leakage.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reproduce v5 group assignment before changing its taxonomy.
v5_classes = sorted(np.unique(original_labels).tolist())
y_split = np.asarray([v5_classes.index(name) for name in original_labels])
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_raw, test_raw = next(outer.split(X, y_split, groups))
inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
inner_train, inner_val = next(
    inner.split(X[train_val_raw], y_split[train_val_raw], groups[train_val_raw])
)
train_raw = train_val_raw[inner_train]
val_raw = train_val_raw[inner_val]

reviewed_mask = ~np.isin(groups, list(UNREVIEWED_GROUPS))
train_idx = train_raw[reviewed_mask[train_raw]]
val_idx = train_val_raw[inner_val][reviewed_mask[train_val_raw[inner_val]]]
test_idx = test_raw[reviewed_mask[test_raw]]

# Active post-v5 taxonomy: plank is rejection/background evidence.
labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)

print("Windows:", X.shape)
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


## Hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class, then equal probability
to each source group inside that class. Windows are uniform inside a group unless
Notebook 13 gives a matching training window a hard-positive weight. Those
modifiers are normalized *within the same class/group*, so neither a long video
nor a group containing many hard positives receives extra total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    groups_per_class = table.groupby("hierarchy_label")["group"].nunique()
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            / groups_per_class.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        # Preserve the v5 baseline perturbation. V6-B will add the complete,
        # explicitly audited augmentation family.
        if self.training and LEGACY_COORDINATE_NOISE_STD:
            sample[:, :, :3] += np.random.normal(
                0, LEGACY_COORDINATE_NOISE_STD, sample[:, :, :3].shape
            ).astype(np.float32)
        sample = sample[:, :, :2].reshape(WINDOW, -1).T
        return torch.from_numpy(sample), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


## Frozen two-stage TCN architecture

This is the same network family used in v5. V6-A tests the sampler and mining
weights, not a larger model. Architecture changes begin only at the later
ST-GCN or conditional multi-scale stages.


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


In [ ]:
def predict_model(model, loader):
    model.eval()
    truths, probabilities = [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE))
            truths.extend(targets.numpy())
            probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities = np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_model(name, num_classes, loaders, seed):
    seed_everything(seed)
    train_loader, val_loader, _ = loaders
    model = SmallTCN(34, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5
    )
    best_path = Path(f"/content/best_v6a_{name}.pt")
    best_f1, stale, history = -1.0, 0, []

    def run_epoch(loader, training):
        model.train(training)
        losses, truths, predictions = [], [], []
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truths.extend(targets.cpu().numpy())
                predictions.extend(logits.argmax(1).cpu().numpy())
        return (
            sum(losses) / len(truths),
            f1_score(truths, predictions, average="macro", zero_division=0),
        )

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(train_loader, True)
        val_loss, val_f1 = run_epoch(val_loader, False)
        scheduler.step(val_f1)
        history.append({
            "model": name, "epoch": epoch,
            "train_loss": train_loss, "train_macro_f1": train_f1,
            "val_loss": val_loss, "val_macro_f1": val_f1,
        })
        print(
            f"{name} {epoch:02d} train loss={train_loss:.4f} F1={train_f1:.3f} "
            f"| val loss={val_loss:.4f} F1={val_f1:.3f}"
        )
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(name, "early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1


gate_model, gate_history, gate_best_f1 = train_model(
    "gate", 2, gate_loaders, SEED
)
exercise_model, exercise_history, exercise_best_f1 = train_model(
    "exercise", len(SUPPORTED_CLASSES), exercise_loaders, SEED + 1
)
print("Best validation F1 — gate:", round(gate_best_f1, 4))
print("Best validation F1 — exercise:", round(exercise_best_f1, 4))


## Interim validation-only threshold

This sweep keeps the v5 evaluation protocol so V6-A can be inspected. It does
**not** complete backbone step 4. After V6-C, a separate creator/person-separated
calibration set will select the production gate threshold and probability
calibration.


In [ ]:
_, _, gate_val_probability = predict_model(gate_model, gate_loaders[1])


def full_probabilities(indices):
    features = X[indices, :, :, :2].reshape(
        len(indices), WINDOW, -1
    ).transpose(0, 2, 1)
    gate_parts, exercise_parts = [], []
    gate_model.eval()
    exercise_model.eval()
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate_model(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise_model(batch).softmax(1).cpu().numpy())
    return np.concatenate(gate_parts), np.concatenate(exercise_parts)


_, exercise_val_probabilities = full_probabilities(val_idx)
exercise_val_predictions = exercise_val_probabilities.argmax(1)
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 37):
    end_predictions = np.asarray([
        SUPPORTED_CLASSES[class_id] if probability >= threshold else "unknown"
        for probability, class_id in zip(
            gate_val_probability[:, 1], exercise_val_predictions
        )
    ])
    supported_mask = labels[val_idx] != "unknown"
    unknown_mask = ~supported_mask
    threshold_rows.append({
        "threshold": float(threshold),
        "validation_macro_f1": float(f1_score(
            labels[val_idx], end_predictions, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
        "supported_accuracy": float(
            (end_predictions[supported_mask] == labels[val_idx][supported_mask]).mean()
        ),
        "unknown_rejection": float(
            (end_predictions[unknown_mask] == "unknown").mean()
        ),
    })
threshold_frame = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_frame.sort_values(
    ["validation_macro_f1", "supported_accuracy"], ascending=False
).iloc[0]
GATE_THRESHOLD = float(best_threshold_row.threshold)
print("Interim validation threshold:", GATE_THRESHOLD)
display(threshold_frame.sort_values("validation_macro_f1", ascending=False).head(10))


## Internal test report and export

The internal group-separated test measures this controlled change. It is not a
complete 14-class external benchmark. Do not run the locked long-video test
until the planned model-selection sequence calls for it.


In [ ]:
gate_test_prob, exercise_test_prob = full_probabilities(test_idx)
gate_test_pred = gate_test_prob.argmax(1)
exercise_test_pred = exercise_test_prob.argmax(1)
end_to_end_pred = np.asarray([
    SUPPORTED_CLASSES[class_id] if probability >= GATE_THRESHOLD else "unknown"
    for probability, class_id in zip(gate_test_prob[:, 1], exercise_test_pred)
])

supported_test_mask = labels[test_idx] != "unknown"
classifier_truth = labels[test_idx][supported_test_mask]
classifier_pred = np.asarray([
    SUPPORTED_CLASSES[index] for index in exercise_test_pred[supported_test_mask]
])
metrics = {
    "experiment": "v6a_hierarchical_sampling_hard_positive_weights",
    "active_supported_class_count": len(SUPPORTED_CLASSES),
    "retired_classes_mapped_to_unknown": sorted(RETIRED_CLASSES),
    "training_archives": [path.name for path in zip_paths],
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "sampler": {
        "gate": "binary_class_then_group_then_window_x_hard_positive_weight",
        "exercise": "canonical_class_then_group_then_window",
        "matched_hard_positive_training_windows": len(matched_training_keys),
    },
    "gate_threshold": GATE_THRESHOLD,
    "threshold_selection": "interim group-separated validation; not final calibration",
    "gate": {
        "accuracy": float(accuracy_score(y_gate[test_idx], gate_test_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_gate[test_idx], gate_test_pred)),
        "macro_f1": float(f1_score(
            y_gate[test_idx], gate_test_pred, average="macro", zero_division=0
        )),
    },
    "exercise_classifier_oracle": {
        "accuracy": float(accuracy_score(classifier_truth, classifier_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(classifier_truth, classifier_pred)),
        "macro_f1": float(f1_score(
            classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "end_to_end": {
        "accuracy": float(accuracy_score(labels[test_idx], end_to_end_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(labels[test_idx], end_to_end_pred)),
        "macro_f1": float(f1_score(
            labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
            average="macro", zero_division=0,
        )),
    },
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
}
print(json.dumps(metrics, indent=2))

REPORT_ROOT = Path("/content/calorie_v6a_hierarchical_tcn_report")
if REPORT_ROOT.exists():
    shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()

pd.DataFrame(gate_history + exercise_history).to_csv(
    REPORT_ROOT / "training_history.csv", index=False
)
threshold_frame.to_csv(REPORT_ROOT / "gate_threshold_sweep.csv", index=False)
sampler_audit.to_csv(REPORT_ROOT / "sampler_audit.csv", index=False)
(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2))

split_rows = []
for split_name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    split_rows.extend({
        "window_key": window_keys[index],
        "group_id": groups[index],
        "clip_id": clip_ids[index],
        "source_dataset": sources[index],
        "original_class": original_labels[index],
        "active_class": labels[index],
        "split": split_name,
    } for index in indices)
pd.DataFrame(split_rows).to_csv(REPORT_ROOT / "split_assignments.csv", index=False)

reports = {
    "gate_classification_report.json": classification_report(
        y_gate[test_idx], gate_test_pred, labels=[0, 1],
        target_names=["unknown", "supported"], output_dict=True, zero_division=0,
    ),
    "exercise_classification_report.json": classification_report(
        classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES,
        output_dict=True, zero_division=0,
    ),
    "end_to_end_classification_report.json": classification_report(
        labels[test_idx], end_to_end_pred, labels=ALL_CLASSES,
        output_dict=True, zero_division=0,
    ),
}
for filename, report in reports.items():
    (REPORT_ROOT / filename).write_text(json.dumps(report, indent=2))

plots = [
    ("gate", y_gate[test_idx], gate_test_pred, [0, 1], ["unknown", "supported"]),
    ("exercise", classifier_truth, classifier_pred, SUPPORTED_CLASSES, SUPPORTED_CLASSES),
    ("end_to_end", labels[test_idx], end_to_end_pred, ALL_CLASSES, ALL_CLASSES),
]
for name, truth, prediction, matrix_labels, tick_labels in plots:
    matrix = confusion_matrix(truth, prediction, labels=matrix_labels)
    plt.figure(figsize=(max(6, len(tick_labels) * 0.75), max(5, len(tick_labels) * 0.65)))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues",
        xticklabels=tick_labels, yticklabels=tick_labels,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(name.replace("_", " ").title() + " test confusion matrix")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

history_frame = pd.DataFrame(gate_history + exercise_history)
for name, group in history_frame.groupby("model"):
    group.plot(
        x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4),
        grid=True, title=name,
    )
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_learning_curve.png", dpi=180)
    plt.show()

pd.DataFrame({
    "window_key": window_keys[test_idx],
    "clip_id": clip_ids[test_idx],
    "source_dataset": sources[test_idx],
    "true_class": labels[test_idx],
    "gate_supported_probability": gate_test_prob[:, 1],
    "exercise_prediction": [SUPPORTED_CLASSES[index] for index in exercise_test_pred],
    "exercise_confidence": exercise_test_prob.max(axis=1),
    "predicted_class": end_to_end_pred,
}).to_csv(REPORT_ROOT / "test_window_predictions.csv", index=False)

torch.save({
    "gate_state_dict": gate_model.state_dict(),
    "exercise_state_dict": exercise_model.state_dict(),
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
    "retired_classes": sorted(RETIRED_CLASSES),
    "gate_threshold": GATE_THRESHOLD,
    "window_frames": WINDOW,
    "input_features": 34,
    "architecture": "two_stage_small_tcn_v6a",
    "experiment": metrics["experiment"],
}, REPORT_ROOT / "two_stage_tcn_v6a.pt")

zip_path = shutil.make_archive(
    "/content/calorie_v6a_hierarchical_tcn_report", "zip", root_dir=REPORT_ROOT
)
print("Report ZIP:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)


## Next controlled step

Return `calorie_v6a_hierarchical_tcn_report.zip`. We will compare it with v5,
audit the sampler totals and split assignments, and only then implement V6-B's
train-only augmentation. Do not add V6-C features or decoder changes to this
run.
